In [ ]:
%matplotlib inline

import pandas as pd
from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = '30-7a'
well_name = '30-7a-7'
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()
    well_data = project.get_well_data(well_name)

# Lithology Estimation

In [ ]:
import matplotlib.pyplot as plt

from quick_pp.lithology.sand_shale import SandShale
from quick_pp.lithology.carbonate import Carbonate
from quick_pp.porosity import *
from quick_pp.qaqc import *
from quick_pp.plotter.plotter import plotly_log, neutron_density_xplot
from quick_pp.rock_type import estimate_vsh_gr
from quick_pp.utils import *

## Process sandstone interval

In [ ]:
# Clean up data
well_data = badhole_flagging(well_data)

for col in ['GR', 'RT', 'NPHI', 'RHOB']:
    well_data.loc[:, col] = remove_straights(well_data[col])

# SandShale
ss_mask = well_data.model == 'sandshale'
model_data = well_data[ss_mask]
# Initialize lithology model
args = {
    'litho_model': 'ss',
    # 'dry_clay_point': (.3, 2.7),
    'wet_clay_point': (0.42, 2.44),
    'hc_corr_angle': neu_den_xplot_hc_correction_angle(rho_water=1.0, rho_hc=0.8, HI_hc=0.9),
    'hc_buffer': 0.01,
}

ss_model = SandShale(**args)
vsand, vcld, _ = ss_model.estimate_lithology(
    nphi=model_data['NPHI'], rhob=model_data['RHOB']
)
args.update(ss_model.__dict__)
# well.update_config(args)  # Save lithology model to well

# Choose to skip HC correction or not
skip_hc_correction = False
if skip_hc_correction is True:
    nphihc, rhobhc = model_data['NPHI'], model_data['RHOB']
else:
    # Implement hydrocarbon correction
    vsh_gr = estimate_vsh_gr(model_data['GR'])
    nphihc, rhobhc, hc_flag = neu_den_xplot_hc_correction(
        model_data['NPHI'], model_data['RHOB'],
        dry_min1_point=args['dry_sand_point'],
        dry_clay_point=args['dry_clay_point'],
        corr_angle=args['hc_corr_angle'], buffer=args['hc_buffer']
    )
    
    # Correct density log
    rhob_corr = den_correction(nphihc, model_data['GR'], vsh_gr=vsh_gr, alpha=0.1)
    badhole_flag =  np.where(abs(model_data['RHOB'] - rhob_corr) > 0.2, 1, 0)
    rhob_corr = np.where((badhole_flag == 1) & (hc_flag == 0), rhob_corr, rhobhc)

    # Estimate lithology
    ss_model = SandShale(**args)
    vsand, vcld, _ = ss_model.estimate_lithology(
        nphi=nphihc, rhob=rhob_corr,
    )

In [ ]:
neutron_density_xplot(model_data['NPHI'], model_data['RHOB'], dry_min1_point=args['dry_sand_point'], **args)

In [ ]:
neutron_density_xplot(nphihc, rhobhc, dry_min1_point=args['dry_sand_point'], **args)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import numpy as np

from quick_pp.rock_type import estimate_vsh_gr

# Estimate porosity
phit = neu_den_xplot_poro(
    nphihc, rhobhc, model='ss',
    dry_min1_point=args['dry_sand_point'],
    dry_clay_point=args['dry_clay_point'],
)

# PHID needs unnormalized lithology
rho_ma = rho_matrix(vsand=vsand, vclay=vcld)
phid = density_porosity(rhobhc, rho_ma)

# Fill missing values in phit with phid
phit = np.where(np.isnan(phit), phid, phit)

# Normalize lithology
volumes = dict(vcld=vcld, vsand=vsand)
volumes = normalize_volumetric(phit, **volumes)
vcld, vsand = volumes['vcld'], volumes['vsand']

# Calculate vclb: volume of clay bound water and phie
phit_shale = estimate_shale_porosity(model_data.NPHI, phid)
vclb = vcld * phit_shale
vclay = vcld + vclb

phie = phit - vclb

well_data.loc[ss_mask, 'NPHI_HC'] = nphihc
well_data.loc[ss_mask, 'RHOB_HC'] = rhobhc
well_data.loc[ss_mask, 'VCLAY'] = vclay
well_data.loc[ss_mask, 'VSAND'] = vsand
well_data.loc[ss_mask, 'PHIT'] = phit
well_data.loc[ss_mask, 'PHID'] = phid

vsh_gr_1 = estimate_vsh_gr(model_data['GR'])
fig, axs = plt.subplots(3, 1, figsize=(20, 5), sharex=True)
axs[0].plot(model_data.DEPTH, phit, label='PHIT')
axs[0].plot(model_data.DEPTH, phid, label='PHID')
axs[0].scatter(model_data.DEPTH, model_data.CPORE, label='CPORE' , marker='.', color='black')
axs[0].set_ylim(0, .5)
axs[0].legend()

axs[1].plot(model_data.DEPTH, vsh_gr_1, label='vsh_gr_1')
axs[1].plot(model_data.DEPTH, vcld, label='vcld')
axs[1].plot(model_data.DEPTH, vclay, label='vclay')
axs[1].set_ylim(-.1, 1.1)
axs[1].legend()

axs[2].plot(model_data.DEPTH, model_data['GR'], label='GR')
axs[2].set_ylim(0,250)
axs[2].legend()

## Process carbonate interval

In [ ]:
# Carbonate
carb_mask = well_data.model == 'carbonate'
model_data = well_data[carb_mask]

carbonate_type =  'limestone'  # 'dolostone'  #
model = 'single'  # 'double'  #
method = 'neu_den'  # 'pef_den'  #

# Initialize lithology model
args = {
    'litho_model': 'carb',
    'dry_calc_point': (.0, 2.71),
    'wet_clay_point': (0.33, 2.44),
    'sw_water_salinity': 15000,
    'sw_m': 1.85,
    'sw_n': 1.85,
    'hc_corr_angle': neu_den_xplot_hc_correction_angle(rho_water=1.0, rho_hc=0.8, HI_hc=0.9),
    'hc_buffer': 0.01,
    'ressum_cutoffs': dict(
        VSHALE=.5,
        PHIT=0,
        SWT=1
    )
}

vsh_gr = estimate_vsh_gr(model_data['GR'])
carb_model = Carbonate(**args)
_, _, _ = carb_model.estimate_lithology(
    nphi=model_data['NPHI'], rhob=model_data['RHOB'], vsh_gr=vsh_gr,
    model=model, method=method, carbonate_type=carbonate_type
)
args.update(carb_model.__dict__)
# well.update_config(args)  # Save lithology model to well

# Choose to skip HC correction or not
skip_hc_correction = False
if skip_hc_correction is True:
    nphihc, rhobhc = model_data['NPHI'], model_data['RHOB']
else:
    # Implement hydrocarbon correction
    nphihc, rhobhc, hc_flag = neu_den_xplot_hc_correction(
        model_data['NPHI'], model_data['RHOB'],
        dry_min1_point=args['dry_calc_point'],
        dry_clay_point=args['dry_clay_point'],
        corr_angle=args['hc_corr_angle'], buffer=args['hc_buffer']
    )
    
    # Correct density log
    rhob_corr = den_correction(nphihc, model_data['GR'], vsh_gr=vsh_gr, alpha=0.1)
    badhole_flag =  np.where(abs(model_data['RHOB'] - rhob_corr) > 0.2, 1, 0)
    rhob_corr = np.where((badhole_flag == 1) & (hc_flag == 0), rhob_corr, rhobhc)

    # Estimate lithology
    carb_model = Carbonate(**args)
    vclay, vcalc, vdolo = carb_model.estimate_lithology(
        nphi=nphihc, rhob=rhob_corr, vsh_gr=vsh_gr,
        model=model, method=method, carbonate_type=carbonate_type
    )

In [ ]:
neutron_density_xplot(model_data['NPHI'], model_data['RHOB'], dry_min1_point=args['dry_calc_point'], **args)

In [ ]:
neutron_density_xplot(nphihc, rhobhc, dry_min1_point=args['dry_calc_point'], **args)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import numpy as np

from quick_pp.rock_type import estimate_vsh_gr

# Estimate porosity
phit = neu_den_xplot_poro(
    nphihc, rhobhc, model='carb',
    dry_min1_point=args['dry_calc_point'],
    dry_clay_point=args['dry_clay_point'],
)

# PHID needs unnormalized lithology
rho_ma = rho_matrix(vclay=vclay, vcalc=vcalc, vdolo=vdolo)
phid = density_porosity(rhobhc, rho_ma)

# Fill missing values in phit with phid
phit = np.where(np.isnan(phit), phid, phit)

# Normalize lithology
volumes = dict(vclay=vclay, vcalc=vcalc, vdolo=vdolo)
volumes = normalize_volumetric(phit, **volumes)
vclay, vcalc, vdolo = volumes['vclay'], volumes['vcalc'], volumes['vdolo']

# Calculate vclb: volume of clay bound water and phie
phit_shale = estimate_shale_porosity(model_data.NPHI, phid)
vclb = vclay * phit_shale

phie = phit - vclb

well_data.loc[carb_mask, 'NPHI_HC'] = nphihc
well_data.loc[carb_mask, 'RHOB_HC'] = rhobhc
well_data.loc[carb_mask, 'VCLAY'] = vclay
well_data.loc[carb_mask, 'VCALC'] = vcalc
well_data.loc[carb_mask, 'PHIT'] = phit
well_data.loc[carb_mask, 'PHID'] = phid

vsh_gr_1 = estimate_vsh_gr(model_data['GR'])
fig, axs = plt.subplots(3, 1, figsize=(20, 5), sharex=True)
axs[0].plot(model_data.DEPTH, phit, label='PHIT')
axs[0].plot(model_data.DEPTH, phid, label='PHID')
axs[0].scatter(model_data.DEPTH, model_data.CPORE, label='CPORE' , marker='.', color='black')
axs[0].set_ylim(0, .5)
axs[0].legend()

axs[1].plot(model_data.DEPTH, vsh_gr_1, label='vsh_gr_1')
axs[1].plot(model_data.DEPTH, vclay, label='vclay')
axs[1].set_ylim(-.1, 1.1)
axs[1].legend()

axs[2].plot(model_data.DEPTH, model_data['GR'], label='GR')
axs[2].set_ylim(0,250)
axs[2].legend()

# Plotting the result

In [ ]:
# Plot the results
fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
fig.show(config=dict(scrollZoom=True))
# fig.write_html(rf"{well_name}_log.html", config=dict(scrollZoom=True))

# Apply to all

In [ ]:
final_args = {}
for well_name, well_data in df.groupby('WELL_NAME'):

    # Clean up data
    well_data = badhole_flagging(well_data)

    for col in ['GR', 'RT', 'NPHI', 'RHOB']:
        well_data.loc[:, col] = remove_straights(well_data[col])

    ### SandShale
    ss_mask = well_data.model == 'sandshale'
    model_data = well_data[ss_mask]
    # Initialize lithology model
    args = {
        'litho_model': 'ss',
        # 'dry_clay_point': (.3, 2.7),
        'wet_clay_point': (0.42, 2.44),
        'sw_water_salinity': 15000,
        'sw_m': 1.85,
        'sw_n': 1.85,
        'hc_corr_angle': neu_den_xplot_hc_correction_angle(rho_water=1.0, rho_hc=0.8, HI_hc=0.9),
        'hc_buffer': 0.01,
        'ressum_cutoffs': dict(
            VSHALE=.5,
            PHIT=0,
            SWT=1
        )
    }

    ss_model = SandShale(**args)
    vsand, vcld, _ = ss_model.estimate_lithology(
        nphi=model_data['NPHI'], rhob=model_data['RHOB']
    )
    args.update(ss_model.__dict__)

    # Implement hydrocarbon correction
    vsh_gr = estimate_vsh_gr(model_data['GR'])
    nphihc, rhobhc, hc_flag = neu_den_xplot_hc_correction(
        model_data['NPHI'], model_data['RHOB'],
        dry_min1_point=args['dry_sand_point'],
        dry_clay_point=args['dry_clay_point'],
        corr_angle=args['hc_corr_angle'], buffer=args['hc_buffer']
    )
    
    # Correct density log
    rhob_corr = den_correction(nphihc, model_data['GR'], vsh_gr=vsh_gr, alpha=0.1)
    badhole_flag =  np.where(abs(model_data['RHOB'] - rhob_corr) > 0.2, 1, 0)
    rhob_corr = np.where((badhole_flag == 1) & (hc_flag == 0), rhob_corr, rhobhc)

    # Estimate lithology
    ss_model = SandShale(**args)
    vsand, vcld, _ = ss_model.estimate_lithology(
        nphi=nphihc, rhob=rhob_corr,
    )
    
    # Estimate porosity
    phit = neu_den_xplot_poro(
        nphihc, rhobhc, model='ss',
        dry_min1_point=args['dry_sand_point'],
        dry_clay_point=args['dry_clay_point'],
    )

    # PHID needs unnormalized lithology
    rho_ma = rho_matrix(vsand=vsand, vclay=vcld)
    phid = density_porosity(rhobhc, rho_ma)
    phit = np.where(np.isnan(phit), phid, phit)

    # Normalize lithology
    volumes = dict(vcld=vcld, vsand=vsand)
    volumes = normalize_volumetric(phit, **volumes)
    vcld, vsand = volumes['vcld'], volumes['vsand']

    # Calculate vclb: volume of clay bound water and phie
    phit_shale = estimate_shale_porosity(model_data.NPHI, phid)
    vclb = vcld * phit_shale
    vclay = vcld + vclb
    phie = phit - vclb

    final_args['sandshale'] = args
    well_data.loc[ss_mask, 'NPHI_HC'] = nphihc
    well_data.loc[ss_mask, 'RHOB_HC'] = rhobhc
    well_data.loc[ss_mask, 'VCLAY'] = vclay
    well_data.loc[ss_mask, 'VSAND'] = vsand
    well_data.loc[ss_mask, 'PHIT'] = phit
    well_data.loc[ss_mask, 'PHID'] = phid
    well_data.loc[ss_mask, 'PHIE'] = phie


    ### Carbonate
    carb_mask = well_data.model == 'carbonate'
    model_data = well_data[carb_mask]

    carbonate_type =  'limestone'  # 'dolostone'  #
    model = 'single'  # 'double'  #
    method = 'neu_den'  # 'pef_den'  #

    # Initialize lithology model
    args = {
        'litho_model': 'carb',
        'dry_calc_point': (.0, 2.71),
        'wet_clay_point': (0.33, 2.44),
        'hc_corr_angle': neu_den_xplot_hc_correction_angle(rho_water=1.0, rho_hc=0.8, HI_hc=0.9),
        'hc_buffer': 0.01,
    }
    vsh_gr = estimate_vsh_gr(model_data['GR'])
    carb_model = Carbonate(**args)
    _, _, _ = carb_model.estimate_lithology(
        nphi=model_data['NPHI'], rhob=model_data['RHOB'], vsh_gr=vsh_gr,
        model=model, method=method, carbonate_type=carbonate_type
    )
    args.update(carb_model.__dict__)
    
    # Implement hydrocarbon correction
    nphihc, rhobhc, hc_flag = neu_den_xplot_hc_correction(
        model_data['NPHI'], model_data['RHOB'],
        dry_min1_point=args['dry_calc_point'],
        dry_clay_point=args['dry_clay_point'],
        corr_angle=args['hc_corr_angle'], buffer=args['hc_buffer']
    )
    
    # Correct density log
    rhob_corr = den_correction(nphihc, model_data['GR'], vsh_gr=vsh_gr, alpha=0.1)
    badhole_flag =  np.where(abs(model_data['RHOB'] - rhob_corr) > 0.2, 1, 0)
    rhob_corr = np.where((badhole_flag == 1) & (hc_flag == 0), rhob_corr, rhobhc)

    # Estimate lithology
    carb_model = Carbonate(**args)
    vclay, vcalc, vdolo = carb_model.estimate_lithology(
        nphi=nphihc, rhob=rhob_corr, vsh_gr=vsh_gr,
        model=model, method=method, carbonate_type=carbonate_type
    )

    # Estimate porosity
    phit = neu_den_xplot_poro(
        nphihc, rhobhc, model='carb',
        dry_min1_point=args['dry_calc_point'],
        dry_clay_point=args['dry_clay_point'],
    )

    # PHID needs unnormalized lithology
    rho_ma = rho_matrix(vclay=vclay, vcalc=vcalc, vdolo=vdolo)
    phid = density_porosity(rhobhc, rho_ma)
    phit = np.where(np.isnan(phit), phid, phit)

    # Normalize lithology
    volumes = dict(vclay=vclay, vcalc=vcalc, vdolo=vdolo)
    volumes = normalize_volumetric(phit, **volumes)
    vclay, vcalc, vdolo = volumes['vclay'], volumes['vcalc'], volumes['vdolo']

    # Calculate vclb: volume of clay bound water and phie
    phit_shale = estimate_shale_porosity(model_data.NPHI, phid)
    vclb = vclay * phit_shale
    phie = phit - vclb

    final_args['carbonate'] = args
    well_data.loc[carb_mask, 'NPHI_HC'] = nphihc
    well_data.loc[carb_mask, 'RHOB_HC'] = rhobhc
    well_data.loc[carb_mask, 'VCLAY'] = vclay
    well_data.loc[carb_mask, 'VCALC'] = vcalc
    well_data.loc[carb_mask, 'PHIT'] = phit
    well_data.loc[carb_mask, 'PHID'] = phid
    well_data.loc[carb_mask, 'PHIE'] = phie

    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(well_data, well_configs={well_name: final_args})
        project.save()

## PHIT vs. CPORE Validation

Initial comparison indicates a poor match with MAPE of around 30%. The core points in

In [ ]:
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    score_df = project.get_all_data()
score_df = score_df[['WELL_NAME', 'CPORE', 'PHIT']].copy()
score_df.dropna(inplace=True)
mape = round(mean_absolute_percentage_error(score_df.CPORE, score_df.PHIT), 2)
r2 = round(r2_score(score_df.CPORE, score_df.PHIT), 2)
print(f"\n ### PHIT MAPE: {mape:.2f}")
print(f" ### PHIT R2: {r2:.2f}")

plt.scatter(score_df.CPORE, score_df.PHIT, label=f'Overall - R2: {r2}, MAPE: {mape}')
for well, data in score_df.groupby('WELL_NAME'):
    mape = round(mean_absolute_percentage_error(data.CPORE, data.PHIT), 2)
    r2 = round(r2_score(data.CPORE, data.PHIT), 2)
    plt.scatter(data.CPORE, data.PHIT, label=f'{well} - R2: {r2}, MAPE: {mape}')
plt.xlabel('Actual')
plt.ylabel('Calculated')
plt.xlim(0, .5)
plt.ylim(0, .5)
plt.legend()

In [ ]:
STOP

In [ ]:
from dtw import dtw, rabinerJuangStepPattern

df = project.get_all_data()

shift_summaries = []
return_df = pd.DataFrame()
for well, data in df.groupby('WELL_NAME'):
    core_data = data[['DEPTH', 'CORE_ID', 'CPORE', 'CPERM']].dropna().sort_values('DEPTH').reset_index(drop=True)
    
    df_corrected = data.copy()
    if len(core_data) > 0:
        log_data = data[['DEPTH', 'PHIT']].dropna().sort_values('DEPTH').reset_index(drop=True)

        core_indices_in_phit = np.searchsorted(log_data.DEPTH, core_data.DEPTH, side='left')
        all_indices = []
        window = 1
        for idx in core_indices_in_phit:
            # Define start and end points for slicing
            start_idx = max(0, idx - window) # Use max to avoid negative index
            end_idx = min(len(log_data.PHIT), idx + window + 1) # Use min to avoid going out of bounds
            # Store the indices to use for DTW later if needed
            indices_to_add = list(range(start_idx, end_idx))
            all_indices.extend(indices_to_add)
        unique_indices = sorted(list(set(all_indices)))
        log_data = log_data.iloc[unique_indices].reset_index(drop=True)

        # Extract the numpy arrays for the DTW algorithm
        phit_vals = log_data['PHIT'].values
        cpore_vals = core_data['CPORE'].values

        alignment = dtw(cpore_vals, phit_vals,
                        distance_only=False,
                        keep_internals=True,
                        step_pattern="symmetric2")

        # The alignment object contains the mapping between indices
        core_indices = alignment.index1  # Indices for the `core_data` DataFrame
        log_indices = alignment.index2   # Indices for the `log_data` DataFrame

        # Create a detailed map from the alignment, including values needed for comparison
        correction_map_df = pd.DataFrame({
            'core_idx': core_indices,
            'log_idx': log_indices,
            'ORIGINAL_DEPTH': core_data.loc[core_indices, 'DEPTH'].values,
            'CORE_ID_SHIFTED': core_data.loc[core_indices, 'CORE_ID'].values,
            'CPORE_SHIFTED': core_data.loc[core_indices, 'CPORE'].values,
            'CPERM_SHIFTED': core_data.loc[core_indices, 'CPERM'].values,
            'MATCHED_PHIT': log_data.loc[log_indices, 'PHIT'].values,
            'DEPTH_CORRECTED': log_data.loc[log_indices, 'DEPTH'].values
        })

        # Calculate the absolute porosity difference for each potential match
        correction_map_df['PORO_DIFF'] = (correction_map_df['CPORE_SHIFTED'] - correction_map_df['MATCHED_PHIT']).abs()
        correction_map_df['DEPTH_SHIFT'] = round(correction_map_df.DEPTH_CORRECTED - correction_map_df.ORIGINAL_DEPTH, 3)
        correction_map_df = correction_map_df.sort_values('PORO_DIFF').drop_duplicates('ORIGINAL_DEPTH')
        # final_correction = correction_map_df.sort_values('PORO_DIFF').drop_duplicates('DEPTH_CORRECTED')

        if not correction_map_df.empty:
            summary = correction_map_df[['CORE_ID_SHIFTED', 'ORIGINAL_DEPTH', 'DEPTH_CORRECTED', 'DEPTH_SHIFT']].copy()
            summary.rename(columns={'CORE_ID_SHIFTED': 'CORE_ID'}, inplace=True)
            summary['WELL_NAME'] = well
            # Reorder columns for clarity
            summary = summary[['WELL_NAME', 'CORE_ID', 'ORIGINAL_DEPTH', 'DEPTH_CORRECTED', 'DEPTH_SHIFT']]
            shift_summaries.append(summary)

        df_corrected = pd.merge(
            data,
            correction_map_df[['DEPTH_CORRECTED', 'CORE_ID_SHIFTED', 'CPORE_SHIFTED', 'CPERM_SHIFTED']],
            left_on='DEPTH',
            right_on='DEPTH_CORRECTED',
            how='left'
        )
    return_df = pd.concat([return_df, df_corrected])
merged_df = return_df.copy()

# After the loop, combine all summaries into a single DataFrame
if shift_summaries:
    final_summary_df = pd.concat(shift_summaries, ignore_index=True)
    final_summary_df.to_csv(r'data/core_depth_shifts.csv', index=False)

In [ ]:
# Compare the score between shifted and non shifted.
score_df = merged_df[['WELL_NAME', 'CPORE', 'CPORE_SHIFTED', 'PHIT']].copy()
score_df.dropna(inplace=True)

mape = round(mean_absolute_percentage_error(score_df.CPORE, score_df.PHIT), 2)
r2 = round(r2_score(score_df.CPORE, score_df.PHIT), 2)
plt.scatter(score_df.CPORE, score_df.PHIT, label=f'Overall Original Data - R2: {r2}, MAPE: {mape}')

mape = round(mean_absolute_percentage_error(score_df.CPORE_SHIFTED, score_df.PHIT), 2)
r2 = round(r2_score(score_df.CPORE_SHIFTED, score_df.PHIT), 2)
plt.scatter(score_df.CPORE_SHIFTED, score_df.PHIT, label=f'Shifted Data - R2: {r2}, MAPE: {mape}')

for well, data in score_df.groupby('WELL_NAME'):
    mape = round(mean_absolute_percentage_error(data.CPORE_SHIFTED, data.PHIT), 2)
    r2 = round(r2_score(data.CPORE_SHIFTED, data.PHIT), 2)
    plt.scatter(data.CPORE_SHIFTED, data.PHIT, label=f'{well} - R2: {r2}, MAPE: {mape}')
plt.xlabel('Actual')
plt.ylabel('Calculated')
plt.xlim(0, .5)
plt.ylim(0, .5)
plt.legend()

In [ ]:
copy_df = merged_df[merged_df.WELL_NAME == '15-9-19-BT2']
plt.figure(figsize=(20, 3))
plt.scatter(copy_df.DEPTH, copy_df.CPORE, c='green', marker='.', label='CPORE ORI')
plt.scatter(copy_df.DEPTH, copy_df.CPORE_SHIFTED, c='r', marker='x', label='CPORE SHIFTED')
plt.plot(copy_df.DEPTH, copy_df.PHIT, 'b--', label='PHIT')

# --- Annotation Loop ---
# Iterate over each row in the DataFrame to add annotations
cores = copy_df.dropna(subset='CORE_ID_SHIFTED')
for index, row in cores.iterrows():
    plt.annotate(
        text=row['CORE_ID_SHIFTED'],                      # The text to display (the CORE_ID)
        xy=(row['DEPTH'], row['CPORE_SHIFTED']),  # The point to annotate (x, y)
        xytext=(5, 5),                            # The position of the text (x, y offset)
        textcoords='offset points',               # Interpret xytext as an offset from the point
        ha='left',                                # Horizontal alignment
        va='bottom',                              # Vertical alignment
        fontsize=8,
        arrowprops=dict(arrowstyle='->', color='gray') # Optional: adds an arrow
    )
# --- End of Annotation Loop ---

plt.legend()
plt.ylim(0, .5)
plt.xlim(4035, 4110)

In [ ]:
return_df = pd.DataFrame()
for well, data in merged_df.groupby('WELL_NAME'):
    # 1. Find all unique, non-null IDs from the 'CORE_ID_SHIFTED' column.
    ori_core_ids = data['CORE_ID'].dropna().unique()
    shifted_core_ids = data['CORE_ID_SHIFTED'].dropna().unique()

    # 2. Create a boolean mask for rows where 'CORE_ID' is in the list of shifted IDs.
    mask_shifted = data['CORE_ID'].isin(shifted_core_ids)
    mask_ori = data['CORE_ID_SHIFTED'].isin(ori_core_ids)

    # 3. Use the mask_shifted with .loc to update the 'CPORE' and 'CPERM' columns.
    # For the rows where the condition is True, we assign the values from the '_SHIFTED' columns.
    data.loc[mask_shifted, ['CORE_ID', 'CPORE', 'CPERM']] = np.nan
    data.loc[mask_ori, ['CORE_ID', 'CPORE', 'CPERM']] = data.loc[mask_ori, ['CORE_ID_SHIFTED', 'CPORE_SHIFTED', 'CPERM_SHIFTED']].values
    return_df = pd.concat([return_df, data])
merged_df = return_df.copy()

In [ ]:
copy_df = merged_df[merged_df.WELL_NAME == '15-9-19-BT2']
plt.figure(figsize=(25, 2))
plt.scatter(copy_df.DEPTH, copy_df.CPORE, c='green', marker='.', label='CPORE ORI')
plt.scatter(copy_df.DEPTH, copy_df.CPORE_SHIFTED, c='r', marker='x', label='CPORE SHIFTED')
plt.plot(copy_df.DEPTH, copy_df.PHIT, 'b--', label='PHIT')

# --- Annotation Loop ---
# Iterate over each row in the DataFrame to add annotations
cores = copy_df.dropna(subset='CORE_ID')
for index, row in cores.iterrows():
    plt.annotate(
        text=row['CORE_ID'],                      # The text to display (the CORE_ID)
        xy=(row['DEPTH'], row['CPORE']),  # The point to annotate (x, y)
        xytext=(5, 5),                            # The position of the text (x, y offset)
        textcoords='offset points',               # Interpret xytext as an offset from the point
        ha='left',                                # Horizontal alignment
        va='bottom',                              # Vertical alignment
        fontsize=8,
        arrowprops=dict(arrowstyle='->', color='gray') # Optional: adds an arrow
    )
# --- End of Annotation Loop ---

plt.legend()
plt.ylim(0, .4)
# plt.xlim(4035, 4110)
plt.xlim(4035, 4050)

In [ ]:
# Save the proposed core depth correction
project.update_data(merged_df)
project.save()